<a href="https://colab.research.google.com/github/SYNGASBH/beardstyle-app/blob/main/Microwave_Plasma_Gasification_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# New Section

In [ ]:
import FreeCAD as App
import Part
import math

def generate_parametrized_reactor(power_kw=30.0, input_kg_h=20.0):
    """
    Generiše geometriju reaktora na osnovu snage plazme i kapaciteta.
    Logika:
    1. Unutrašnji prečnik (D_in) korelira sa snagom (cca 10mm po 1kW za stabilan vortex).
    2. Debljina obloge (L_ref) raste sa snagom radi termičke izolacije.
    3. Vanjski prečnik (D_out) je suma unutrašnjeg prečnika i dvostruke obloge.
    """

    doc = App.newDocument("MW_VORTEX_Reactor_V2")

    # --- PROCESNA LOGIKA / DIMENZIONISANJE ---
    # Osnovni unutrašnji radijus (Fluid Zone) - minimalno 50mm, raste sa snagom
    inner_radius = 50.0 + (power_kw * 1.5)

    # Debljina vatrostalne obloge (Al2O3-SiC + izolacija) [cite: 29, 30]
    # Što je veća snaga, potrebna je bolja izolacija da Inox 316L ostane na <200°C
    refractory_thickness = 80.0 + (power_kw * 0.8)

    # Visina reaktora - bazirana na vremenu zadržavanja (Residence time) [cite: 12]
    reactor_height = 400.0 + (input_kg_h * 5.0)

    # Debljina čeličnog plašta (Inox 316L) prema EN 13445 (pojednostavljeno) [cite: 16, 17]
    steel_thickness = 6.0 if power_kw < 50 else 8.0

    print(f"Generisanje reaktora za {power_kw}kW i {input_kg_h}kg/h...")
    print(f"Unutrašnji radijus: {inner_radius} mm")
    print(f"Debljina obloge: {refractory_thickness} mm")

    # --- GEOMETRIJA ---

    # 1. Fluid Zone (Unutrašnjost gdje je plazma)
    fluid_cyl = Part.makeCylinder(inner_radius, reactor_height)
    fluid_obj = doc.addObject("Part::Feature", "FluidZone")
    fluid_obj.Shape = fluid_cyl
    fluid_obj.ViewObject.Transparency = 70

    # 2. Vatrostalna obloga (Refractory Layer) [cite: 27]
    refr_outer_radius = inner_radius + refractory_thickness
    refr_cyl = Part.makeCylinder(refr_outer_radius, reactor_height)
    refr_shape = refr_cyl.cut(fluid_cyl)
    refr_obj = doc.addObject("Part::Feature", "RefractoryWall")
    refr_obj.Shape = refr_shape
    refr_obj.ViewObject.ShapeColor = (0.8, 0.7, 0.5)

    # 3. Vanjska Inox 316L ljuska (Steel Shell) [cite: 16, 25]
    steel_outer_radius = refr_outer_radius + steel_thickness
    steel_cyl = Part.makeCylinder(steel_outer_radius, reactor_height)
    steel_shape = steel_cyl.cut(refr_cyl)
    steel_obj = doc.addObject("Part::Feature", "SteelShell_316L")
    steel_obj.Shape = steel_shape
    steel_obj.ViewObject.ShapeColor = (0.7, 0.7, 0.8)

    # 4. Kvarcna cijev (Quartz Tube) - standard za WR-340 [cite: 19, 21]
    # Pozicionirana na dnu gdje mikrovalovi ulaze
    quartz_radius = 40.0 # Standardno za torch sisteme
    quartz_height = 150.0
    quartz_cyl = Part.makeCylinder(quartz_radius, quartz_height, App.Vector(0,0,-50))
    quartz_obj = doc.addObject("Part::Feature", "QuartzTube")
    quartz_obj.Shape = quartz_cyl
    quartz_obj.ViewObject.ShapeColor = (0.9, 0.9, 1.0)
    quartz_obj.ViewObject.Transparency = 50

    doc.recompute()
    return doc

def export_to_step(doc, filename="SYNGAS_Reactor_Export"):
    """ Izvozi sve vidljive objekte u STEP format za CFD/FEA [cite: 44, 56] """
    objs = doc.Objects
    Part.export(objs, filename + ".step")
    print(f"Model uspješno izvezen u {filename}.step")

if __name__ == "__main__":
    # Testni pokret: 30kW magnetron, 20kg/h otpada
    new_doc = generate_parametrized_reactor(power_kw=30.0, input_kg_h=20.0)
    # export_to_step(new_doc)

ModuleNotFoundError: No module named 'FreeCAD'

In [ ]:
# Čisti inženjerski proračun energetske bilance (bez AI poziva)
def analiza_energetike_syngas_bh():
    snaga_ulaz = 40.0 # kW
    kapacitet = 25.0 # kg/h
    sec_potreban = 1.5 # kWh/kg (specifična energija iz literature)
    gubitak_vortex = 0.05 # 5% gubitaka

    # 1. Energija potrebna za proces (zagrijavanje + reakcije)
    energija_proces = kapacitet * sec_potreban # 25 * 1.5 = 37.5 kW

    # 2. Neto dostupna snaga nakon toplinskih gubitaka
    # Stari sustav (500mm) bi imao 30% gubitaka, vortex ima 5%
    neto_snaga = snaga_ulaz * (1 - gubitak_vortex) # 40 * 0.95 = 38 kW

    bilanca = neto_snaga - energija_proces

    print(f"--- ANALIZA ENERGETSKE DOSTATNOSTI (SYNGAS BH) ---")
    print(f"Instalirana snaga (4 magnetrona): {snaga_ulaz} kW")
    print(f"Neto snaga (uz vortex hlađenje): {neto_snaga} kW")
    print(f"Potrebna snaga za 25 kg/h otpada: {energija_proces} kW")

    if bilanca >= 0:
        print(f"STATUS: POZITIVNO (Višak: {bilanca:.2f} kW)")
        print("Zaključak: 40 kW je dovoljno za održavanje 1250°C u jezgri od 150 mm.")
    else:
        print(f"STATUS: NEGATIVNO (Deficit: {abs(bilanca):.2f} kW)")

analiza_energetike_syngas_bh()

--- ANALIZA ENERGETSKE DOSTATNOSTI (SYNGAS BH) ---
Instalirana snaga (4 magnetrona): 40.0 kW
Neto snaga (uz vortex hlađenje): 38.0 kW
Potrebna snaga za 25 kg/h otpada: 37.5 kW
STATUS: POZITIVNO (Višak: 0.50 kW)
Zaključak: 40 kW je dovoljno za održavanje 1250°C u jezgri od 150 mm.


koje su preporuke

In [1]:
import math

def proracun_snage_reaktora(kapacitet_kg_h=20.0, temp_radna=1500.0, vlaga_procentat=15.0):
    """
    Racuna potrebnu mikrovalnu snagu (kW) za SYNGAS BH reaktor.
    """
    # 1. Konstante
    temp_ambijent = 20.0       # °C
    cp_otpad = 1.4             # kJ/(kg*K) - prosjek za komunalni otpad
    cp_voda = 4.18             # kJ/(kg*K)
    h_evap = 2260.0            # kJ/kg - latentna toplota isparavanja vode

    # 2. Proračun mase
    masa_sec = kapacitet_kg_h / 3600.0   # kg/s
    masa_vode = masa_sec * (vlaga_procentat / 100.0)
    masa_suha = masa_sec - masa_vode

    # 3. Energija za zagrijavanje suhe materije
    Q_suho = masa_suha * cp_otpad * (temp_radna - temp_ambijent)

    # 4. Energija za vodu (zagrijavanje do 100°C + isparavanje + pregrijavanje pare)
    Q_voda_zagrijavanje = masa_vode * cp_voda * (100 - temp_ambijent)
    Q_voda_isparavanje = masa_vode * h_evap
    Q_para_pregrijavanje = masa_vode * 2.0 * (temp_radna - 100) # cp pare cca 2.0
    Q_voda_total = Q_voda_zagrijavanje + Q_voda_isparavanje + Q_para_pregrijavanje

    # 5. Ukupna termička snaga
    P_termička = Q_suho + Q_voda_total

    # 6. Efikasnost sistema (Gubici kroz zidove + Efikasnost magnetrona)
    # Refraktorni gubici su cca 15%, efikasnost MW apsorpcije cca 85%
    efikasnost_prenosa = 0.70
    snaga_magnetrona = P_termička / efikasnost_prenosa

    print("-" * 30)
    print(f"REZULTATI PRORAČUNA ZA SYNGAS BH")
    print("-" * 30)
    print(f"Kapacitet:          {kapacitet_kg_h} kg/h")
    print(f"Radna temperatura:  {temp_radna} °C")
    print(f"Potrebna MW snaga:  {snaga_magnetrona:.2f} kW")
    print("-" * 30)
    print(f"NAPOMENA: Koristi {math.ceil(snaga_magnetrona)} kW kao ulaz u FreeCAD.")

    return snaga_magnetrona

# Pokretanje proračuna
potrebna_snaga = proracun_snage_reaktora(kapacitet_kg_h=20.0, temp_radna=1500.0)

------------------------------
REZULTATI PRORAČUNA ZA SYNGAS BH
------------------------------
Kapacitet:          20.0 kg/h
Radna temperatura:  1500.0 °C
Potrebna MW snaga:  20.40 kW
------------------------------
NAPOMENA: Koristi 21 kW kao ulaz u FreeCAD.
